# Part 2 – Summary statistics, Results & Discussion, Webpage files
**Paste these cells at the END of your notebook** (after the trend/export sections).
They use variables you already created: `YEARS, SOURCE, ndbi_images, ndbi_difference, city, city_geom, city_shp,
wards, results_df, counts_df, ward_out, NDBI_VIS, DIFF_VIS, OUT_DIR, CENTER_LAT, CENTER_LON, gif_final`.

## 15a. Area with NDBI > 0 for every year (area change)
NDBI > 0 is a common rule-of-thumb for "built-up / bare-surface-like" pixels. It is a heuristic, not a calibrated built-up class – say so in the report.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

NDBI_THRESHOLD = 0.0
px_km2 = ee.Image.pixelArea().divide(1e6)

area_rows = []
for y in YEARS:
    img = ndbi_images[y]
    stack_a = (px_km2.updateMask(img.mask()).rename("valid_km2")
               .addBands(px_km2.updateMask(img.gt(NDBI_THRESHOLD)).rename("high_km2")))
    s = stack_a.reduceRegion(
        reducer=ee.Reducer.sum(), geometry=city_geom, scale=20,
        maxPixels=1e13, tileScale=4
    ).getInfo()
    area_rows.append({"Year": y,
                      "Valid_area_km2": s["valid_km2"],
                      "NDBI_gt0_km2": s["high_km2"]})

area_df = pd.DataFrame(area_rows)
area_df["NDBI_gt0_pct_of_valid"] = 100 * area_df["NDBI_gt0_km2"] / area_df["Valid_area_km2"]
display(area_df.round(2))

## 15b. Where did NDBI change? Area of increase / decrease / stable (2025 − 2018)
`CHANGE_T` is the change you treat as meaningful. 0.10 is a conservative choice (it filters out much of the composite/seasonal noise) – state it in your methods and, if you like, test 0.05.

In [ ]:
CHANGE_T = 0.10
d = ndbi_difference

chg_img = (px_km2.updateMask(d.gt(CHANGE_T)).rename("increase")
           .addBands(px_km2.updateMask(d.lt(-CHANGE_T)).rename("decrease"))
           .addBands(px_km2.updateMask(d.abs().lte(CHANGE_T)).rename("stable")))

chg_area = chg_img.reduceRegion(
    reducer=ee.Reducer.sum(), geometry=city_geom, scale=20,
    maxPixels=1e13, tileScale=4
).getInfo()

chg_stats = d.reduceRegion(
    reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
    geometry=city_geom, scale=20, maxPixels=1e13, tileScale=4
).getInfo()

chg_total = sum(chg_area.values())
chg = {
    "increase_km2": chg_area["increase"],
    "decrease_km2": chg_area["decrease"],
    "stable_km2":   chg_area["stable"],
    "increase_pct": 100 * chg_area["increase"] / chg_total,
    "decrease_pct": 100 * chg_area["decrease"] / chg_total,
    "stable_pct":   100 * chg_area["stable"]   / chg_total,
    "pixel_mean_change": chg_stats["NDBI_change_mean"],
    "pixel_std_change":  chg_stats["NDBI_change_stdDev"],
}
display(pd.DataFrame([chg]).T.rename(columns={0: "value"}).round(3))

## 15c. City-wide summary: average annual mean, 2018→2025 difference, trend

In [ ]:
r = results_df.sort_values("Year").reset_index(drop=True)

m_first = r.loc[r["Year"] == YEARS[0], "Mean_NDBI"].iloc[0]
m_last  = r.loc[r["Year"] == YEARS[-1], "Mean_NDBI"].iloc[0]
abs_change = m_last - m_first
pct_change = 100 * abs_change / abs(m_first) if abs(m_first) >= 0.05 else np.nan   # % is meaningless near zero

slope, intercept, r_val, p_val, se = stats.linregress(r["Year"], r["Mean_NDBI"])
hi = r.loc[r["Mean_NDBI"].idxmax()]
lo = r.loc[r["Mean_NDBI"].idxmin()]

S = {
    "avg_annual_mean": r["Mean_NDBI"].mean(),         # "average annual mean" NDBI, 2018-2025
    "std_annual_mean": r["Mean_NDBI"].std(),
    "first_year": YEARS[0], "last_year": YEARS[-1],
    "m_first": m_first, "m_last": m_last,
    "abs_change": abs_change, "pct_change": pct_change,
    "range_annual": r["Mean_NDBI"].max() - r["Mean_NDBI"].min(),
    "slope_per_year": slope, "r2": r_val ** 2, "p_value": p_val,
    "hi_year": int(hi["Year"]), "hi_val": hi["Mean_NDBI"],
    "lo_year": int(lo["Year"]), "lo_val": lo["Mean_NDBI"],
}

summary_table = pd.DataFrame([
    ["Average of annual mean NDBI (2018–2025)", f"{S['avg_annual_mean']:.4f}"],
    ["Std. deviation of annual means",           f"{S['std_annual_mean']:.4f}"],
    [f"City mean NDBI {YEARS[0]}",               f"{m_first:.4f}"],
    [f"City mean NDBI {YEARS[-1]}",              f"{m_last:.4f}"],
    [f"Difference {YEARS[-1]} − {YEARS[0]}",     f"{abs_change:+.4f}"],
    ["Percent change (relative to |baseline|)",
        f"{pct_change:+.1f} %" if not np.isnan(pct_change) else "n/a (baseline close to zero)"],
    ["Linear trend (NDBI per year)",             f"{slope:+.4f} (R² = {S['r2']:.2f}, p = {p_val:.3f})"],
    ["Highest annual mean",                      f"{S['hi_year']} ({S['hi_val']:.4f})"],
    ["Lowest annual mean",                       f"{S['lo_year']} ({S['lo_val']:.4f})"],
    [f"Area NDBI > {NDBI_THRESHOLD} in {YEARS[0]}",
        f"{area_df.iloc[0]['NDBI_gt0_km2']:.1f} km² ({area_df.iloc[0]['NDBI_gt0_pct_of_valid']:.1f} %)"],
    [f"Area NDBI > {NDBI_THRESHOLD} in {YEARS[-1]}",
        f"{area_df.iloc[-1]['NDBI_gt0_km2']:.1f} km² ({area_df.iloc[-1]['NDBI_gt0_pct_of_valid']:.1f} %)"],
    [f"Area with NDBI increase > {CHANGE_T}", f"{chg['increase_km2']:.1f} km² ({chg['increase_pct']:.1f} %)"],
    [f"Area with NDBI decrease < −{CHANGE_T}", f"{chg['decrease_km2']:.1f} km² ({chg['decrease_pct']:.1f} %)"],
    ["Area with stable NDBI",                  f"{chg['stable_km2']:.1f} km² ({chg['stable_pct']:.1f} %)"],
], columns=["Metric", "Value"])

display(summary_table)
summary_table.to_csv(os.path.join(OUT_DIR, "Faridabad_NDBI_summary.csv"), index=False)
area_df.to_csv(os.path.join(OUT_DIR, "Faridabad_NDBI_area_by_year.csv"), index=False)

## 15d. Ward ranking (largest increase / decrease / highest NDBI)

In [ ]:
w = ward_out.drop(columns="geometry").copy()
for c in ["NDBI_2018", "NDBI_2025", "NDBI_change"]:
    w[c] = pd.to_numeric(w[c], errors="coerce")

name_cols = [c for c in w.columns if "name" in c.lower()]
LABEL = name_cols[0] if name_cols else "ward_id"
cols = [LABEL, "NDBI_2018", "NDBI_2025", "NDBI_change"]

top_inc = w.nlargest(5, "NDBI_change")[cols]
top_dec = w.nsmallest(5, "NDBI_change")[cols]
top_hi  = w.nlargest(5, "NDBI_2025")[cols]

print("Ward label column used:", LABEL)
print("\nLargest NDBI increase:");  display(top_inc.round(3))
print("Largest NDBI decrease:");    display(top_dec.round(3))
print("Highest NDBI in 2025:");     display(top_hi.round(3))

top_inc.to_csv(os.path.join(OUT_DIR, "Faridabad_wards_top_increase.csv"), index=False)
top_dec.to_csv(os.path.join(OUT_DIR, "Faridabad_wards_top_decrease.csv"), index=False)

## 16. Results & Discussion text (built from YOUR numbers)
The paragraphs below are generated from the statistics above, so the numbers always match your outputs.
Lines starting with **`>>> EDIT`** are places where only you can describe what you see on the maps – replace them,
and re-read every generated sentence against your maps before submitting.

In [ ]:
NEGLIGIBLE = 0.01       # |city-mean change| below this is described as "little net change"

def names(df, n=3):
    return ", ".join(str(v) for v in df[LABEL].head(n))

def build_discussion(S, area_df, chg, top_inc, top_dec, top_hi, sources, threshold, change_t):
    y0, y1 = S["first_year"], S["last_year"]

    if abs(S["abs_change"]) < NEGLIGIBLE:
        verb = "showed little net change"
    elif S["abs_change"] > 0:
        verb = "increased"
    else:
        verb = "decreased"

    pct_txt = ("" if np.isnan(S["pct_change"])
               else f", {S['pct_change']:+.1f} % relative to the {y0} value")
    sig = "statistically significant" if S["p_value"] < 0.05 else "not statistically significant"
    a0, a1 = area_df.iloc[0], area_df.iloc[-1]

    # ---- 1. temporal pattern ----
    temporal = [
        f"The city-wide mean Jan–Mar NDBI {verb} from {S['m_first']:.3f} in {y0} to {S['m_last']:.3f} in {y1} "
        f"({S['abs_change']:+.3f}{pct_txt}). "
        f"The average of the eight annual means is {S['avg_annual_mean']:.3f} "
        f"(standard deviation {S['std_annual_mean']:.3f}). "
        f"The highest annual mean occurred in {S['hi_year']} ({S['hi_val']:.3f}) and the lowest in "
        f"{S['lo_year']} ({S['lo_val']:.3f}).",

        f"A linear fit to the annual means gives {S['slope_per_year']:+.4f} NDBI per year "
        f"(R² = {S['r2']:.2f}, p = {S['p_value']:.3f}), which is {sig} at the 5 % level; with only eight "
        f"annual points this should be read as indicative, not conclusive.",
    ]
    if S["range_annual"] > abs(S["abs_change"]):
        temporal.append(
            f"The spread between the highest and lowest annual mean ({S['range_annual']:.3f}) is larger than the "
            f"net {y0}–{y1} change ({abs(S['abs_change']):.3f}). The series therefore fluctuates from year to year "
            f"(for example with winter haze, soil moisture and crop/fallow conditions) and the endpoint "
            f"difference alone should not be presented as a steady trend."
        )
    temporal.append(
        f"The area with NDBI > {threshold} changed from {a0['NDBI_gt0_km2']:.1f} km² "
        f"({a0['NDBI_gt0_pct_of_valid']:.1f} % of valid pixels) in {y0} to {a1['NDBI_gt0_km2']:.1f} km² "
        f"({a1['NDBI_gt0_pct_of_valid']:.1f} %) in {y1}, a difference of "
        f"{a1['NDBI_gt0_km2'] - a0['NDBI_gt0_km2']:+.1f} km²."
    )

    # ---- 2. spatial pattern ----
    spatial = [
        f"Pixel-level change ({y1} minus {y0}) had a mean of {chg['pixel_mean_change']:+.3f} "
        f"(std. {chg['pixel_std_change']:.3f}). Using a ±{change_t} threshold, {chg['increase_km2']:.1f} km² "
        f"({chg['increase_pct']:.1f} %) of the city shows a clear NDBI increase, {chg['decrease_km2']:.1f} km² "
        f"({chg['decrease_pct']:.1f} %) a clear decrease and {chg['stable_km2']:.1f} km² "
        f"({chg['stable_pct']:.1f} %) is stable.",

        f"At ward level, the largest NDBI increases were in {names(top_inc)}, and the largest decreases in "
        f"{names(top_dec)}. The wards with the highest NDBI in {y1} were {names(top_hi)}.",

        ">>> EDIT: describe the main spatial pattern you see on your maps – where the high-NDBI (red/yellow) "
        "areas are (dense core, industrial belts, city edges), where NDBI is low (green/blue: vegetation, "
        "Aravalli forest patch, Yamuna side, water), and whether the red areas in the difference map are on the "
        "urban fringe (possible expansion) or inside the old core. Only mention features you can see.",
    ]

    # ---- 3. limitations ----
    lim = [
        "Spatial resolution: B8 is 10 m but B11 (SWIR1) is 20 m, so NDBI is effectively a 20 m product; mixed "
        "pixels (buildings, trees, bare soil in one pixel) blur the signal, especially in small wards.",
        "Temporal resolution and compositing: each year is one median of all clear Jan–Mar scenes (about a "
        "5-day revisit, fewer clear scenes in hazy weeks). A median removes clouds and noise but hides changes "
        "within the season.",
        "Cloud, haze and fog: winter smog and fog in the Delhi–NCR region can be misclassified by the cloud "
        "mask and shift SWIR/NIR reflectance, creating year-to-year differences unrelated to land cover.",
        "Index ambiguity: NDBI is not specific to buildings. Dry bare soil, fallow fields, dry riverbeds and "
        "quarries also have high NDBI in Jan–Mar, while vegetation and moist surfaces lower it. A rise in NDBI "
        "can therefore mean new construction or vegetation loss/drier ground, and the NDBI > "
        f"{threshold} rule is a heuristic, not a validated built-up map.",
        "No ground validation: results were not compared with a land-cover product or reference imagery, so "
        "area figures should be treated as relative indicators.",
        "Boundary and simplification: the ward boundary was simplified (about 20 m) and the analysis is limited "
        "to it, so edge pixels and anything outside the boundary are not represented.",
    ]
    if "TOA" in sources.values():
        toa_years = ", ".join(str(k) for k, v in sources.items() if v == "TOA")
        lim.insert(2,
            f"Dataset difference: Sentinel-2 surface reflectance (Level-2A) was not available in Earth Engine "
            f"for {toa_years}, so top-of-atmosphere (Level-1C) data were used there. TOA values include "
            f"atmospheric effects, so comparisons involving {toa_years} are less strictly like-for-like.")

    # ---- 4. conclusion ----
    conclusion = (
        f"Between {y0} and {y1} the Jan–Mar city mean NDBI of Faridabad {verb} "
        f"({S['abs_change']:+.3f}), with {chg['increase_pct']:.1f} % of the area showing a clear increase and "
        f"{chg['decrease_pct']:.1f} % a clear decrease. Because NDBI also responds to bare soil and seasonal "
        f"conditions, these results indicate where the built-up/bare spectral signal strengthened or weakened, "
        f"and should be confirmed with higher-resolution imagery or a land-cover dataset before concluding "
        f"that the changes are urban growth."
    )

    return [
        ("Temporal change", temporal),
        ("Spatial pattern", spatial),
        ("Limitations", [("ul", lim)]),
        ("Conclusion", [conclusion]),
    ]

discussion = build_discussion(S, area_df, chg, top_inc, top_dec, top_hi, SOURCE, NDBI_THRESHOLD, CHANGE_T)

def discussion_to_md(sections):
    out = ["# Results and Discussion\n"]
    for head, items in sections:
        out.append(f"## {head}\n")
        for it in items:
            if isinstance(it, tuple) and it[0] == "ul":
                out.extend(f"- {b}" for b in it[1]); out.append("")
            else:
                out.append(it + "\n")
    return "\n".join(out)

def discussion_to_html(sections):
    import html as _h
    out = []
    for head, items in sections:
        out.append(f"<h3>{_h.escape(head)}</h3>")
        for it in items:
            if isinstance(it, tuple) and it[0] == "ul":
                out.append("<ul>" + "".join(f"<li>{_h.escape(b)}</li>" for b in it[1]) + "</ul>")
            else:
                cls = ' class="todo"' if it.startswith(">>>") else ""
                out.append(f"<p{cls}>{_h.escape(it)}</p>")
    return "\n".join(out)

discussion_md = discussion_to_md(discussion)
with open(os.path.join(OUT_DIR, "Faridabad_NDBI_discussion.md"), "w", encoding="utf-8") as f:
    f.write(discussion_md)

print(discussion_md)

## 17a. Static thematic maps (title, legend, units, boundary) – these never expire
The Earth Engine tile links in the interactive HTML files stop working after some hours, so a hosted page needs
image-based outputs. These maps are saved as PNG files.

In [ ]:
import requests
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import LinearSegmentedColormap
import shutil

SITE_DIR   = os.path.join(OUT_DIR, "site")
FRAMES_DIR = os.path.join(SITE_DIR, "frames")
os.makedirs(FRAMES_DIR, exist_ok=True)

outline = (ee.Image().byte().paint(city, 1, 2)
           .visualize(palette=["000000", "000000"], min=0, max=1))
minx, miny, maxx, maxy = city_shp.bounds
extent = [minx, maxx, miny, maxy]

def save_thumb(ee_rgb, path, dims=900):
    url = ee_rgb.getThumbURL({"region": city_geom.bounds(), "dimensions": dims,
                              "format": "png", "crs": "EPSG:4326"})
    resp = requests.get(url, timeout=300)
    resp.raise_for_status()
    with open(path, "wb") as fh:
        fh.write(resp.content)
    return path

def make_cmap(vis):
    return LinearSegmentedColormap.from_list("c", ["#" + c for c in vis["palette"]])

def thematic_map(png_path, title, vis, cbar_label, out_path):
    fig, ax = plt.subplots(figsize=(7.5, 8))
    ax.imshow(Image.open(png_path), extent=extent)
    ax.set_aspect(1 / np.cos(np.radians(CENTER_LAT)))
    ax.set_title(title, fontsize=13, weight="bold")
    ax.set_xlabel("Longitude (°E)"); ax.set_ylabel("Latitude (°N)")
    sm = mpl.cm.ScalarMappable(cmap=make_cmap(vis),
                               norm=mpl.colors.Normalize(vis["min"], vis["max"]))
    fig.colorbar(sm, ax=ax, orientation="horizontal", fraction=0.04, pad=0.10, label=cbar_label)
    ax.annotate("N", xy=(0.95, 0.95), xytext=(0.95, 0.86), xycoords="axes fraction",
                ha="center", fontsize=14, weight="bold",
                arrowprops=dict(facecolor="black", width=3, headwidth=10))
    fig.text(0.5, 0.01, "Sentinel-2 | 20 m | Faridabad boundary in black", ha="center", fontsize=8)
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close(fig)

# yearly frames (also used by the permanent time slider) + thematic maps for first and last year
for y in YEARS:
    frame = ndbi_images[y].visualize(**NDBI_VIS).unmask(255).blend(outline)
    save_thumb(frame, os.path.join(FRAMES_DIR, f"ndbi_{y}.png"))
    print("saved frame", y)

for y in (YEARS[0], YEARS[-1]):
    thematic_map(os.path.join(FRAMES_DIR, f"ndbi_{y}.png"),
                 f"Faridabad NDBI, Jan–Mar {y} ({SOURCE[y]})", NDBI_VIS,
                 "NDBI (unitless, SWIR1 − NIR)/(SWIR1 + NIR)",
                 os.path.join(SITE_DIR, f"map_ndbi_{y}.png"))

diff_frame = ndbi_difference.visualize(**DIFF_VIS).unmask(255).blend(outline)
diff_png = save_thumb(diff_frame, os.path.join(FRAMES_DIR, "ndbi_difference.png"))
thematic_map(diff_png, f"Change in NDBI, Jan–Mar {YEARS[-1]} minus {YEARS[0]}", DIFF_VIS,
             "NDBI change (unitless)", os.path.join(SITE_DIR, "map_ndbi_difference.png"))

# ward choropleth (static)
ward_out_num = ward_out.copy()
for c in ["NDBI_2018", "NDBI_2025", "NDBI_change"]:
    ward_out_num[c] = pd.to_numeric(ward_out_num[c], errors="coerce")

fig, axes = plt.subplots(1, 3, figsize=(20, 7.5))
for ax, col, vis, ttl in [
    (axes[0], "NDBI_2018",   NDBI_VIS, f"Ward mean NDBI, Jan–Mar {YEARS[0]}"),
    (axes[1], "NDBI_2025",   NDBI_VIS, f"Ward mean NDBI, Jan–Mar {YEARS[-1]}"),
    (axes[2], "NDBI_change", DIFF_VIS, f"Ward mean NDBI change ({YEARS[-1]} − {YEARS[0]})"),
]:
    ward_out_num.plot(column=col, cmap=make_cmap(vis), vmin=vis["min"], vmax=vis["max"],
                      edgecolor="black", linewidth=0.4, ax=ax,
                      missing_kwds={"color": "lightgrey"})
    ax.set_title(ttl, fontsize=12, weight="bold"); ax.set_axis_off()
    sm = mpl.cm.ScalarMappable(cmap=make_cmap(vis),
                               norm=mpl.colors.Normalize(vis["min"], vis["max"]))
    fig.colorbar(sm, ax=ax, orientation="horizontal", fraction=0.04, pad=0.03, label="NDBI (unitless)")
fig.savefig(os.path.join(SITE_DIR, "map_ward_choropleth.png"), dpi=200, bbox_inches="tight")
plt.close(fig)

# trend chart + GIF into the site folder
shutil.copy(os.path.join(OUT_DIR, "Faridabad_Mean_NDBI_trend.png"), os.path.join(SITE_DIR, "trend.png"))
shutil.copy(gif_final, os.path.join(SITE_DIR, "ndbi_timelapse.gif"))
print("Static maps ready in", SITE_DIR)

## 17b. Permanent time slider (uses the saved PNG frames – works on any web host)

In [ ]:
SLIDER_STATIC = r"""<!DOCTYPE html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Faridabad NDBI time slider</title>
<style>
body{margin:0;font-family:Arial,sans-serif;text-align:center;background:#fff;color:#111}
#bar{padding:10px;background:#f3f4f6}
#yearLabel{font-size:18px;font-weight:bold;margin:0 12px}
#yearSlider{width:min(380px,55%);vertical-align:middle}
button{padding:6px 14px;font-size:14px;cursor:pointer}
#frame{max-width:100%;max-height:78vh;margin-top:8px}
.gradient{width:260px;height:12px;margin:6px auto 0;
 background:linear-gradient(to right,#0000FF,#00FFFF,#00FF00,#FFFF00,#FF0000)}
.lbl{width:260px;margin:0 auto;display:flex;justify-content:space-between;font-size:12px}
</style></head><body>
<div id="bar"><button id="play">▶ Play</button>
<input id="yearSlider" type="range" min="0" max="__LAST__" step="1" value="0">
<span id="yearLabel"></span></div>
<img id="frame" alt="NDBI map">
<div class="gradient"></div>
<div class="lbl"><span>-0.5</span><span>NDBI</span><span>+0.5</span></div>
<script>
const years = __YEARS__, src = __SRC__;
const img = document.getElementById("frame"), slider = document.getElementById("yearSlider");
const label = document.getElementById("yearLabel"), btn = document.getElementById("play");
years.forEach(y => { const p = new Image(); p.src = "frames/ndbi_" + y + ".png"; });   // preload
let cur = 0, timer = null;
function show(i){ cur = i; img.src = "frames/ndbi_" + years[i] + ".png";
  label.textContent = "Jan–Mar " + years[i] + " (" + src[String(years[i])] + ")"; slider.value = i; }
function stop(){ if(timer){clearInterval(timer); timer=null;} btn.textContent="▶ Play"; }
slider.oninput = function(){ stop(); show(Number(this.value)); };
btn.onclick = function(){ if(timer){stop();} else { btn.textContent="⏸ Pause";
  timer = setInterval(()=>show((cur+1)%years.length), 1200);} };
show(0);
</script></body></html>
"""

slider_static = (SLIDER_STATIC
    .replace("__LAST__", str(len(YEARS) - 1))
    .replace("__YEARS__", json.dumps(YEARS))
    .replace("__SRC__", json.dumps({str(y): ("SR" if SOURCE[y] == "SR" else "L1C TOA") for y in YEARS})))

with open(os.path.join(SITE_DIR, "ndbi_slider.html"), "w", encoding="utf-8") as f:
    f.write(slider_static)
print("Saved", os.path.join(SITE_DIR, "ndbi_slider.html"))

## 17c. Portfolio page `index.html` (Question 2)

In [ ]:
import html as _html
import shutil

def img_block(file, caption):
    return (f'<figure><img src="{file}" alt="{_html.escape(caption)}">'
            f'<figcaption>{_html.escape(caption)}</figcaption></figure>')

yearly_tbl = (results_df.merge(area_df, on="Year")
              [["Year", "Source", "Mean_NDBI", "NDBI_gt0_km2", "NDBI_gt0_pct_of_valid"]]
              .rename(columns={"Mean_NDBI": "Mean NDBI", "NDBI_gt0_km2": "Area NDBI>0 (km²)",
                               "NDBI_gt0_pct_of_valid": "Share NDBI>0 (%)"})
              .round(3).to_html(index=False, border=0, classes="tbl"))
summary_html = summary_table.to_html(index=False, border=0, classes="tbl")

PAGE = r"""<!DOCTYPE html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Question 2 – Faridabad NDBI (Jan–Mar, 2018–2025)</title>
<style>
body{font-family:Arial,sans-serif;max-width:1100px;margin:0 auto;padding:0 18px 40px;line-height:1.55;color:#111}
header{background:#172554;color:#fff;margin:0 -18px 20px;padding:18px}
h1{margin:0;font-size:26px} h2{border-bottom:2px solid #172554;padding-bottom:4px;margin-top:34px}
figure{margin:14px 0;text-align:center} figure img{max-width:100%} figcaption{font-size:13px;color:#444}
.tbl{border-collapse:collapse;margin:10px 0} .tbl th,.tbl td{border:1px solid #ccc;padding:5px 10px;text-align:left}
.tbl th{background:#f3f4f6} iframe{width:100%;height:640px;border:1px solid #ccc}
.todo{background:#fff3cd;padding:6px 10px;border-left:4px solid #f0ad4e}
.small{font-size:13px;color:#555}
</style></head><body>
<header><h1>Question 2: Faridabad built-up spectral signal (NDBI)</h1>
<div>Sentinel-2 | same Jan–Mar window every year | 2018–2025 | Google Earth Engine</div></header>

<h2>1. Data and method</h2>
<p>NDBI = (SWIR1 − NIR) / (SWIR1 + NIR) = (B11 − B8) / (B11 + B8), computed per cloud-masked scene from
Sentinel-2 Harmonized imagery (SR where available, Level-1C TOA otherwise) and combined into one median
composite per year using scenes from 1 January to 31 March. The same colour scale (−0.5 to +0.5) is used for
every year. Analysis is limited to the Faridabad ward boundary.</p>

<h2>2. Dynamic output</h2>
<h3>Time-lapse GIF</h3>
__GIF__
<h3>Time slider</h3>
<iframe src="ndbi_slider.html" title="NDBI time slider"></iframe>
<p class="small"><a href="ndbi_slider.html">Open the slider in a new tab</a></p>

<h2>3. Thematic maps</h2>
__MAPS__

<h2>4. Summary statistics</h2>
__SUMMARY__
<h3>Annual values</h3>
__YEARLY__
__TREND__

__DISCUSSION__

<h2>Downloads</h2>
<p><a href="Faridabad_NDBI_summary.csv">Summary table (CSV)</a> ·
<a href="Faridabad_Mean_NDBI_2018_2025.csv">Annual mean NDBI (CSV)</a> ·
<a href="Faridabad_ward_NDBI.csv">Ward mean NDBI (CSV)</a></p>
</body></html>
"""

page = (PAGE
    .replace("__GIF__", img_block("ndbi_timelapse.gif", "Annual Jan–Mar NDBI, 2018–2025"))
    .replace("__MAPS__",
             img_block(f"map_ndbi_{YEARS[0]}.png", f"NDBI, Jan–Mar {YEARS[0]}")
             + img_block(f"map_ndbi_{YEARS[-1]}.png", f"NDBI, Jan–Mar {YEARS[-1]}")
             + img_block("map_ndbi_difference.png", f"NDBI change {YEARS[-1]} − {YEARS[0]}")
             + img_block("map_ward_choropleth.png", "Ward-level mean NDBI (choropleth)"))
    .replace("__SUMMARY__", summary_html)
    .replace("__YEARLY__", yearly_tbl)
    .replace("__TREND__", img_block("trend.png", "City mean NDBI trend, 2018–2025"))
    .replace("__DISCUSSION__", "<h2>5. Results and discussion</h2>\n" + discussion_to_html(discussion)))

with open(os.path.join(SITE_DIR, "index.html"), "w", encoding="utf-8") as f:
    f.write(page)

for fname in ["Faridabad_NDBI_summary.csv", "Faridabad_Mean_NDBI_2018_2025.csv", "Faridabad_ward_NDBI.csv"]:
    src_f = os.path.join(OUT_DIR, fname)
    if os.path.exists(src_f):
        shutil.copy(src_f, SITE_DIR)

print("Webpage ready:", os.path.join(SITE_DIR, "index.html"))
print("Open index.html in Chrome to preview. To host, upload the whole 'site' folder (see notes).")